In [ ]:
import numpy as np 
import pandas as pd
pd.set_option('display.max_columns', None)
import polars as pl
import os
import sklearn

import seaborn as sns
import matplotlib.pyplot as plt
import pytz
import plotly.express as px
import plotly.graph_objects as go
import gc
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from xgboost import XGBRegressor
import xgboost as xgb
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.svm import SVR
from sklearn.model_selection import RandomizedSearchCV
from patsy import dmatrices
import joblib
import torch
import holidays
import datetime
import pickle

import warnings
import time
warnings.filterwarnings("ignore")

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern

from sklearn.linear_model import RidgeCV

pd.options.display.float_format = "{:.2f}".format

In [ ]:
print(f"scikit-learn version: {sklearn.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
print(f"joblib version: {joblib.__version__}")
print(f"joblib version: {xgb.__version__}")

In [ ]:
#source: https://www.kaggle.com/code/vitalykudelya/enefit-target-diff
class DataStorage:
    root = "predict-energy-behavior-of-prosumers-1"

    data_cols = [
        "target",
        "prediction_unit_id",
        "county",
        "is_business",
        "product_type",
        "is_consumption",
        "datetime",
        "row_id"
    ]
    client_cols = [ 
        "product_type",
        "county",
        "eic_count",
        "installed_capacity",
        "is_business",
        "date",
    ]
    # gas_prices_cols = ["forecast_date", "lowest_price_per_mwh", "highest_price_per_mwh"]
    # electricity_prices_cols = ["forecast_date", "euros_per_mwh"]
    forecast_weather_cols = [
        "latitude",
        "longitude",
        "hours_ahead",
        "temperature",
        "dewpoint",
        "cloudcover_high",
        "cloudcover_low",
        "cloudcover_mid",
        "cloudcover_total",
        "10_metre_u_wind_component",
        "10_metre_v_wind_component",
        "forecast_datetime",
        "direct_solar_radiation",
        "surface_solar_radiation_downwards",
        "snowfall",
        "total_precipitation",
    ]
    historical_weather_cols = [
        "datetime",
        "temperature",
        "dewpoint",
        "rain",
        "snowfall",
        "surface_pressure",
        "cloudcover_total",
        "cloudcover_low",
        "cloudcover_mid",
        "cloudcover_high",
        "windspeed_10m",
        "winddirection_10m",
        "shortwave_radiation",
        "direct_solar_radiation",
        "diffuse_radiation",
        "latitude",
        "longitude",
    ]
    location_cols = ["longitude", "latitude", "county"]
    target_cols = [
        "target",
        "county",
        "is_business",
        "product_type",
        "is_consumption",
        "datetime",
    ]

    def __init__(self):
        self.df_data = pl.read_csv(
            os.path.join(self.root, "train.csv"),
            columns=self.data_cols,
            try_parse_dates=True,
        )
        self.df_client = pl.read_csv(
            os.path.join(self.root, "client.csv"),
            columns=self.client_cols,
            try_parse_dates=True,
        )
        # self.df_gas_prices = pl.read_csv(
        #     os.path.join(self.root, "gas_prices.csv"),
        #     columns=self.gas_prices_cols,
        #     try_parse_dates=True,
        # )
        # self.df_electricity_prices = pl.read_csv(
        #     os.path.join(self.root, "electricity_prices.csv"),
        #     columns=self.electricity_prices_cols,
        #     try_parse_dates=True,
        # )
        self.df_forecast_weather = pl.read_csv(
            os.path.join(self.root, "forecast_weather.csv"),
            columns=self.forecast_weather_cols,
            try_parse_dates=True,
        )
        self.df_historical_weather = pl.read_csv(
            os.path.join(self.root, "historical_weather.csv"),
            columns=self.historical_weather_cols,
            try_parse_dates=True,
        )
        self.df_weather_station_to_county_mapping = pl.read_csv(
            os.path.join(self.root, "weather_station_to_county_mapping.csv"),
            columns=self.location_cols,
            try_parse_dates=True,
        )
        
        self.df_data = self.df_data.filter(
            pl.col("datetime") >= pd.to_datetime("2021-01-01")
        )
        self.df_target = self.df_data.select(self.target_cols)

        self.schema_data = self.df_data.schema
        self.schema_client = self.df_client.schema
        # self.schema_gas_prices = self.df_gas_prices.schema
        # self.schema_electricity_prices = self.df_electricity_prices.schema
        self.schema_forecast_weather = self.df_forecast_weather.schema
        self.schema_historical_weather = self.df_historical_weather.schema
        self.schema_target = self.df_target.schema

        self.df_weather_station_to_county_mapping = (
            self.df_weather_station_to_county_mapping.with_columns(
                pl.col("latitude").cast(pl.datatypes.Float32),
                pl.col("longitude").cast(pl.datatypes.Float32),
            )
        )
        
        self.df_weather_station_to_county_mapping = self.df_weather_station_to_county_mapping.with_columns(
            pl.col("county").fill_null(12)
        )

    def update_with_new_data(
        self,
        df_new_client,
        # df_new_gas_prices,
        # df_new_electricity_prices,
        df_new_forecast_weather,
        df_new_historical_weather,
        df_new_target,
    ):
        df_new_client = pl.from_pandas(
            df_new_client[self.client_cols], schema_overrides=self.schema_client
        )
        # df_new_gas_prices = pl.from_pandas(
        #     df_new_gas_prices[self.gas_prices_cols],
        #     schema_overrides=self.schema_gas_prices,
        # )
        # df_new_electricity_prices = pl.from_pandas(
        #     df_new_electricity_prices[self.electricity_prices_cols],
        #     schema_overrides=self.schema_electricity_prices,
        # )
        df_new_forecast_weather = pl.from_pandas(
            df_new_forecast_weather[self.forecast_weather_cols],
            schema_overrides=self.schema_forecast_weather,
        )
        df_new_historical_weather = pl.from_pandas(
            df_new_historical_weather[self.historical_weather_cols],
            schema_overrides=self.schema_historical_weather,
        )
        df_new_target = pl.from_pandas(
            df_new_target[self.target_cols], schema_overrides=self.schema_target
        )

        self.df_client = pl.concat([self.df_client, df_new_client]).unique(
            ["date", "county", "is_business", "product_type"]
        )
        self.df_gas_prices = pl.concat([self.df_gas_prices, df_new_gas_prices]).unique(
            ["forecast_date"]
        )
        self.df_electricity_prices = pl.concat(
            [self.df_electricity_prices, df_new_electricity_prices]
        ).unique(["forecast_date"])
        self.df_forecast_weather = pl.concat(
            [self.df_forecast_weather, df_new_forecast_weather]
        ).unique(["forecast_datetime", "latitude", "longitude", "hours_ahead"])
        self.df_historical_weather = pl.concat(
            [self.df_historical_weather, df_new_historical_weather]
        ).unique(["datetime", "latitude", "longitude"])
        self.df_target = pl.concat([self.df_target, df_new_target]).unique(
            ["datetime", "county", "is_business", "product_type", "is_consumption"]
        )

    def preprocess_test(self, df_test):
        df_test = df_test.rename(columns={"prediction_datetime": "datetime"})
        df_test = pl.from_pandas(
            df_test[self.data_cols[1:]], schema_overrides=self.schema_data
        )
        return df_test

In [ ]:
#source: https://www.kaggle.com/code/vitalykudelya/enefit-target-diff
class FeaturesGenerator:
    def __init__(self, data_storage):
        self.data_storage = data_storage
        self.estonian_holidays = list(
            holidays.country_holidays("EE", years=range(2021, 2025)).keys()
        )

    def _add_general_features(self, df_features):
        df_features = (
            df_features.with_columns(
                pl.col("datetime").dt.ordinal_day().alias("dayofyear"),
                pl.col("datetime").dt.hour().alias("hour"),
                pl.col("datetime").dt.day().alias("day"),
                pl.col("datetime").dt.month().alias("month"),
                pl.col("datetime").dt.year().alias("year"),
            ).with_columns(         #to handle periodicity
                (np.pi * pl.col("dayofyear") / 183).sin().alias("sin(dayofyear)"),
                (np.pi * pl.col("dayofyear") / 183).cos().alias("cos(dayofyear)"),
                (np.pi * pl.col("hour") / 12).sin().alias("sin(hour)"),
                (np.pi * pl.col("hour") / 12).cos().alias("cos(hour)"),
            )
        )
        return df_features
    
    def is_country_holiday(self, row):
        return (
            datetime.date(row["year"], row["month"], row["day"])
            in self.estonian_holidays
        )

    def _add_holidays_features(self, df_features):
        df_features = df_features.with_columns(
            pl.struct(["year", "month", "day"])
            .apply(self.is_country_holiday)
            .alias("is_country_holiday")
        )
        return df_features

    def _add_client_features(self, df_features):
        df_client = self.data_storage.df_client
        df_features = df_features.join(
            df_client.with_columns(
                (pl.col("date") + pl.duration(days=2)).cast(pl.Date)
            ),
            on=["county", "is_business", "product_type", "date"],
            how="left",
        )
        
        df_features = df_features.join(
            df_client[["county", "is_business", "product_type", "date", "installed_capacity"]].with_columns(
                (pl.col("date") + pl.duration(days=4)).cast(pl.Date)
            ),
            on=["county", "is_business", "product_type", "date"],
            how="left",
            suffix=f"_48h"
        )
        
        
        return df_features

    def _add_forecast_weather_features(self, df_features):
        df_forecast_weather = self.data_storage.df_forecast_weather
        df_weather_station_to_county_mapping = (
            self.data_storage.df_weather_station_to_county_mapping
        )

        #calculating windspeed based on the u and v components:
        #https://disc.gsfc.nasa.gov/information/data-in-action?title=Derive%20Wind%20Speed%20and%20Direction%20With%20MERRA-2%20Wind%20Components

        df_forecast_weather = df_forecast_weather.with_columns(
            (
                (pl.col('10_metre_u_wind_component')**2 + pl.col('10_metre_v_wind_component')**2).sqrt()
            ).alias('windspeed')
        )
        df_forecast_weather = df_forecast_weather.drop('10_metre_u_wind_component', '10_metre_v_wind_component')
        
        #calculating PV panel temperature: https://doi.org/10.1016/j.ecmx.2022.100182 - Skoplaki model
        df_forecast_weather = df_forecast_weather.with_columns(
            (
                pl.col('temperature') + 
                (0.32 / (8.91 + 2.0 * (pl.col('windspeed') / 0.67))) * pl.col('surface_solar_radiation_downwards')
            ).alias('pv_panel_temperature')
        )

        #calculating energy estimate: DOI: 10.22616/ERDev.2021.20.TF372 - formula 1
        df_forecast_weather = df_forecast_weather.with_columns(
            (
                pl.col('surface_solar_radiation_downwards')/1000
            ).alias('energy')
        )

        #Since there are more weather stations in one county I need to aggegate them, that is why we added mean, min and max aggregations
        df_forecast_weather = (
            df_forecast_weather.rename({"forecast_datetime": "datetime"})
            .filter((pl.col("hours_ahead") >= 22) & pl.col("hours_ahead") <= 45)
            .drop("hours_ahead")
            .with_columns(
                pl.col("latitude").cast(pl.datatypes.Float32),
                pl.col("longitude").cast(pl.datatypes.Float32),
            )
            .join(
                df_weather_station_to_county_mapping,
                how="left",
                on=["longitude", "latitude"],
            )
            .drop("longitude", "latitude")
        )

        df_forecast_weather_mean = (
            df_forecast_weather.filter(pl.col("county").is_not_null())
            .group_by("county", "datetime")
            .mean()
        )
        
        df_forecast_weather_min = (
            df_forecast_weather.filter(pl.col("county").is_not_null())
            .group_by("county", "datetime")
            .min()
        )
        
        df_forecast_weather_max = (
            df_forecast_weather.filter(pl.col("county").is_not_null())
            .group_by("county", "datetime")
            .max()
        )
        
        df_features = df_features.join(
                df_forecast_weather_mean,
                on=["county", "datetime"],
                how="left",
                suffix=f"_mean",
            )
        
        df_features = df_features.join(
                df_forecast_weather_min,
                on=["county", "datetime"],
                how="left",
                suffix=f"_min",
            )
        
        df_features = df_features.join(
                df_forecast_weather_max,
                on=["county", "datetime"],
                how="left",
                suffix=f"_max",
            )
        
        return df_features
        
    def _add_historical_weather_features(self, df_features):
        df_historical_weather = self.data_storage.df_historical_weather
        df_weather_station_to_county_mapping = (
            self.data_storage.df_weather_station_to_county_mapping
        )
        
        #df_historical_weather = df_historical_weather.drop('diffuse_radiation', 'direct_solar_radiation', 'dewpoint', 'surface_pressure')
        #Maybe I shouldn't do that
        # df_historical_weather = df_historical_weather.with_columns(
        #     (pl.col("cloudcover_low").apply(lambda x: 1 if x > 0 else 0))
        # )
        # df_historical_weather = df_historical_weather.with_columns(
        #     (pl.col("cloudcover_mid").apply(lambda x: 1 if x > 0 else 0))
        # )
        # df_historical_weather = df_historical_weather.with_columns(
        #     (pl.col("cloudcover_high").apply(lambda x: 1 if x > 0 else 0))
        # )
        
        df_historical_weather = (
            df_historical_weather.with_columns(
                pl.col("latitude").cast(pl.datatypes.Float32),
                pl.col("longitude").cast(pl.datatypes.Float32),
            )
            .join(
                df_weather_station_to_county_mapping,
                how="left",
                on=["longitude", "latitude"],
            )
            .drop("longitude", "latitude")
        )

        df_historical_weather_mean = (
            df_historical_weather.filter(pl.col("county").is_not_null())
            .group_by("county", "datetime")
            .mean()
        )
        
        df_historical_weather_date = (
            df_historical_weather.group_by("datetime").mean().drop("county")
        )
        
        for hours_lag in [1 * 24, 2 * 24]:
            df_features = df_features.join(
                df_historical_weather_mean.with_columns(
                    pl.col("datetime") + pl.duration(hours=hours_lag)
                ),
                on=["county", "datetime"],
                how="left",
                suffix=f"_historical_mean_{hours_lag}h",
            )
            
            df_features = df_features.join(
                df_historical_weather_date.with_columns(
                    pl.col("datetime") + pl.duration(hours=hours_lag)
                ),
                on=["datetime"],
                how="left",
                suffix=f"_historical_grouped_by_date_{hours_lag}h",
            )
        
        #lag values for shortwave radiation and temperature only
        for hours_lag in [14 * 24, 28 * 24]:
            df_features = df_features.join(
                df_historical_weather_mean[["county", "datetime", 'shortwave_radiation', 'temperature']].with_columns(
                    pl.col("datetime") + pl.duration(hours=hours_lag)
                ),
                on=["county", "datetime"],
                how="left",
                suffix=f"_historical_mean_{hours_lag}h",
            )
            
        return df_features
    
    # def add_electricity_features(self, df_features):
    #     df_electricity =  self.data_storage.df_electricity_prices
    #     df_electricity = df_electricity.rename({"forecast_date": "datetime",
    #                                             "euros_per_mwh": "electricity_euros_per_mwh"
    #                                            })
    #     df_features = df_features.join(
    #         df_electricity.with_columns(
    #             (pl.col("datetime") + pl.duration(days=1))
    #         ),
    #         on=["datetime"],
    #         how="left",
    #     )
    #     return df_features
    
    # def add_gas_features(self, df_features):
    #     df_gas =  self.data_storage.df_gas_prices
    #     df_gas = df_gas.rename({"forecast_date": "temp_date", 
    #                            "lowest_price_per_mwh": "gas_lowest_price_per_mwh",
    #                             "highest_price_per_mwh": "gas_highest_price_per_mwh"})
        
    #     df_features = df_features.with_columns(temp_date = pl.col('datetime').cast(pl.Date))
        
    #     df_features = df_features.join(
    #         df_gas.with_columns(
    #             (pl.col("temp_date") + pl.duration(days=1)).cast(pl.Date)
    #         ),
    #         on=["temp_date"],
    #         how="left",
    #     )
        
    #     df_features = df_features.drop('temp_date')
        
    #     df_features = df_features.with_columns(
    #         gas_avg_price_per_mwh = df_features[['gas_lowest_price_per_mwh', 'gas_highest_price_per_mwh']].mean(axis=1)
    #     )
        
    #     return df_features
    
    def _add_target_features(self, df_features):
        df_target = self.data_storage.df_target
        
        #df_target_all_type_sum and df_target_all_county_type_sum are not needed because they make the dataset more complex, also I should cut the number of target lags as well

        for hours_lag in [
            2 * 24,
            3 * 24,
            4 * 24,
            5 * 24,
            6 * 24,
            7 * 24,
            8 * 24,
            9 * 24,
            10 * 24,
            11 * 24,
            12 * 24,
            13 * 24,
            14 * 24,
        ]:
            df_features = df_features.join(
                df_target.with_columns(
                    pl.col("datetime") + pl.duration(hours=hours_lag)
                ).rename({"target": f"target_{hours_lag}h"}),
                on=[
                    "county",
                    "is_business",
                    "product_type",
                    "is_consumption",
                    "datetime",
                ],
                how="left",
            )

        return df_features

    def _reduce_memory_usage(self, df_features):
        df_features = df_features.with_columns(pl.col(pl.Float64).cast(pl.Float32))
        return df_features

    def _drop_columns(self, df_features):
        df_features = df_features.drop(
            "date", "dayofyear"
        )
        return df_features

    def _to_pandas(self, df_features, y):
        cat_cols = ["county","is_business","product_type","is_consumption"]

        if y is not None:
            df_features = pd.concat([df_features.to_pandas(), y.to_pandas()], axis=1)
        else:
            df_features = df_features.to_pandas()

        df_features = df_features.set_index("row_id")
        df_features[cat_cols] = df_features[cat_cols].astype("category")

        return df_features

    def generate_features(self, df_prediction_items):
        if "target" in df_prediction_items.columns:
            df_prediction_items, y = (
                df_prediction_items.drop("target"),
                df_prediction_items.select("target"),
            )
        else:
            y = None

        df_features = df_prediction_items.with_columns(
            pl.col("datetime").cast(pl.Date).alias("date"),
        )

        for add_features in [
            self._add_general_features,
            self._add_client_features,
            self._add_forecast_weather_features,
            self._add_holidays_features,
            self._add_historical_weather_features,
            self._add_target_features,
            # self.add_electricity_features, 
            # self.add_gas_features,
            self._reduce_memory_usage,
            self._drop_columns,
        ]:
            df_features = add_features(df_features)

        df_features = self._to_pandas(df_features, y)

        return df_features

In [ ]:
#source: https://www.kaggle.com/code/vitalykudelya/enefit-target-diff
data_storage = DataStorage()
features_generator = FeaturesGenerator(data_storage=data_storage)

In [ ]:
#source: https://www.kaggle.com/code/vitalykudelya/enefit-target-diff
df_train_features_wf = features_generator.generate_features(data_storage.df_data)
df_train_features_wf = df_train_features_wf[df_train_features_wf['target'].notnull()]

#filtering down only to production!!
df_train_production = df_train_features_wf[df_train_features_wf['is_consumption'] == 0].copy()

In [ ]:
start_date = '2021-09-03 00:00:00'
end_date = '2023-05-30 23:00:00'

start_date = pd.to_datetime(start_date)
end_date = pd.to_datetime(end_date)
df_train_production = df_train_production[(df_train_production['datetime'] >= start_date) & (df_train_production['datetime'] <= end_date)].copy()

In [ ]:
df_train_production.to_csv('df_train_production.csv')

In [ ]:
df_train_production.shape

In [ ]:
df_train_production.dropna(inplace=True)

In [ ]:
df_train_production['unit_target'] = df_train_production['target']/df_train_production['installed_capacity']
df_train_production['unit_target_48h'] = df_train_production['target_48h']/df_train_production['installed_capacity_48h']

In [ ]:
df_train_production.dropna(inplace=True)

In [ ]:
df_train_production.shape

Persistence model - https://www.sciencedirect.com/science/article/abs/pii/S1364032117311620?via%3Dihub

In [ ]:
split_index = int(0.6 * len(df_train_production))  
print(split_index)

df_test_production = df_train_production[split_index:]
df_test_production["datetime"] = pd.to_datetime(df_test_production["datetime"])

df_test_production = df_test_production.sort_values(by=["prediction_unit_id", "datetime"])
df_test_production["persistence_pred"] = df_test_production.groupby("prediction_unit_id")["unit_target"].shift(24)

df_test_production = df_test_production.dropna()
rmse_persistence = root_mean_squared_error(df_test_production["unit_target"], df_test_production["persistence_pred"])

print(f"Persistence Model RMSE: {rmse_persistence}")


In [ ]:
results = {
    'Model': ['Persistence Model'],
    'RMSE': [rmse_persistence]
}

persistence_results_df = pd.DataFrame(results)
persistence_results_df.to_csv('persistence_rmse.csv', index=False)

In [ ]:
df_train_production.drop(columns=['year', 'is_consumption', 'day', 'installed_capacity', 'prediction_unit_id', 'target', 'eic_count'], inplace=True)

In [ ]:
df_train_production.shape

In [ ]:
X_prod = df_train_production.drop(columns=['unit_target', 'datetime'])
y_prod = df_train_production[['unit_target', 'datetime']]

split_index = int(0.6 * len(X_prod))  
print(split_index)

X_train_prod, X_test_prod = X_prod[:split_index], X_prod[split_index:]
y_train_prod, y_test_prod = y_prod[:split_index], y_prod[split_index:]

**Feature Selection**

In [ ]:
model = XGBRegressor(enable_categorical=True, importance_type='gain')
model.fit(X_train_prod, y_train_prod['unit_target'])

In [ ]:
y_test_prod['y_pred'] = model.predict(X_test_prod)
r2 = r2_score(y_test_prod['unit_target'], y_test_prod['y_pred'])
mae = mean_absolute_error(y_test_prod['unit_target'], y_test_prod['y_pred'])

print(f"R2: {r2:.4f}")
print(f"MAE: {mae:.4f}")

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=y_test_prod['datetime'], y=y_test_prod['y_pred'],
                         mode='lines',
                         name='Predicted',
                         line=dict(color='blue')))

fig.add_trace(go.Scatter(x=y_test_prod['datetime'], y=y_test_prod['unit_target'],
                         mode='lines',
                         name='Real',
                         line=dict(color='red')))

fig.update_layout(title='Predicted vs Real Values',
                  xaxis_title='Date',
                  yaxis_title='Production',
                  xaxis=dict(tickformat='%Y-%m-%d %H:%M',
                             tickangle=45),
                  legend=dict(x=0.01, y=0.99))

fig.show()

In [ ]:
top_n = 50

fig, ax = plt.subplots(figsize=(10, 6))
xgb.plot_importance(model, ax=ax, importance_type='gain',
                    max_num_features=top_n, show_values=True)
plt.xlabel('Gain')
plt.title(f'Top {top_n} Most Important Features')
ax.tick_params(axis='y', labelsize=6)

for text in ax.get_yticklabels():
    text.set_fontsize(6) 
    
for text in ax.texts: 
    text.set_fontsize(6) 
plt.show()

In [ ]:
thresholds = np.sort(model.feature_importances_)
results = []

for thresh in thresholds:
    selection = SelectFromModel(model, threshold=thresh, prefit=True)
    select_X_train = selection.transform(X_train_prod)

    selection_model = XGBRegressor(n_estimators=500, max_depth=5, learning_rate=0.01, enable_categorical=True)
    selection_model.fit(select_X_train, y_train_prod['unit_target'])

    select_X_test = selection.transform(X_test_prod)
    predictions = selection_model.predict(select_X_test)
    r_squared = r2_score(y_test_prod['unit_target'], predictions)
    print("Thresh=%.7f, n=%d, R2: %.5f" % (thresh, select_X_train.shape[1], r_squared))

    results.append({
        'Threshold': thresh,
        'Num_Features': select_X_train.shape[1],
        'R2_Score': r_squared
    })

results_df = pd.DataFrame(results)

print(results_df)

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=results_df['Threshold'], y=results_df['R2_Score'],
                         mode='lines',
                         name='Predicted',
                         line=dict(color='blue')))

fig.update_layout(title='R² Score vs. Feature Importance Threshold',
                  xaxis_title='Threshold',
                  yaxis_title='R2 scores')

fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=results_df['Num_Features'], y=results_df['R2_Score'],
                         mode='lines',
                         name='Predicted',
                         line=dict(color='blue')))

fig.update_layout(title='R² Score vs. Number of features based on feature importance',
                  xaxis_title='Number of features',
                  yaxis_title='R2 scores')

fig.show()

In [ ]:
top_n = 25

fig, ax = plt.subplots(figsize=(10, 6))
xgb.plot_importance(model, ax=ax, importance_type='gain',
                    max_num_features=top_n, show_values=True)
plt.xlabel('Gain')
plt.title(f'Top {top_n} Most Important Features')
ax.tick_params(axis='y', labelsize=6)

for text in ax.get_yticklabels():
    text.set_fontsize(6) 
    
for text in ax.texts: 
    text.set_fontsize(6) 
plt.show()

In [ ]:
importance = model.get_booster().get_score(importance_type='gain')

importance_df = pd.DataFrame({'Feature': list(importance.keys()), 'Importance': list(importance.values())})

sorted_importance_df = importance_df.sort_values(by='Importance', ascending=False)

top_features = sorted_importance_df.head(top_n)['Feature'].tolist()

print("Top Features:", top_features)

#Selecting the top N features from the dataset
X_train_prod_reduced = X_train_prod[top_features]
X_test_prod_reduced = X_test_prod[top_features]

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_prod_reduced)
X_test_scaled = scaler.transform(X_test_prod_reduced)

In [ ]:
df_sns = df_train_production[['surface_solar_radiation_downwards', 'surface_solar_radiation_downwards_min', 'is_business', 'unit_target_48h', 'total_precipitation_max', 
                              'month', 'direct_solar_radiation_max', 'sin(hour)', 'product_type', 'total_precipitation', 'hour', 'is_country_holiday', 'cos(dayofyear)', 
                              'total_precipitation_min', 'county', 'cloudcover_total_historical_grouped_by_date_48h', 'cloudcover_total_historical_mean_48h', 'target_168h', 'target_336h', 
                              'cloudcover_low_max', 'cos(hour)', 'cloudcover_low_historical_grouped_by_date_48h', 'direct_solar_radiation', 
                              'cloudcover_high_historical_mean_24h', 'installed_capacity_48h', 'unit_target']]

scaler = StandardScaler()
scaled_df = scaler.fit_transform(df_sns)

scaled_df = pd.DataFrame(scaled_df, columns=df_sns.columns)

corr = scaled_df.corr(method='pearson')

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', annot_kws={'size': 6})
plt.title('Correlation Heatmap')
plt.show()

XGBoost - Hyperparameter tuning

In [ ]:
param_distributions_xgb = {
    'n_estimators': [100, 500, 1000],        #https://doi.org/10.1016/j.jclepro.2018.08.207 for the hyperparamters or at least the idea
    'max_depth': [5, 10, 15],
    'subsample': [0.7, 0.8, 0.9, 1.0]
}

xgb_model = XGBRegressor()

random_search_xgb = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_distributions_xgb,
    n_iter=50,                            
    scoring='neg_mean_squared_error',     
    cv=5,                                 
    random_state=42,
    n_jobs=-1                             
)

random_search_xgb.fit(X_train_scaled, y_train_prod['unit_target'])

print("Best Parameters:", random_search_xgb.best_params_)
print("Best CV Score:", random_search_xgb.best_score_)

best_model_xgb = random_search_xgb.best_estimator_
test_score_xgb = best_model_xgb.score(X_test_scaled, y_test_prod)
print("Test Score:", test_score_xgb)

Ridge Regression - Hyperparameter tuning

In [ ]:
alphas1 = [0.001, 0.01, 0.1, 1, 10, 100]
ridge_coarse = RidgeCV(alphas=alphas1, store_cv_values=True)
ridge_coarse.fit(X_train_scaled, y_train_prod['unit_target'])
print(f"Best alpha from first search: {ridge_coarse.alpha_}")

alphas2 = list(range(1, 100))
ridge_fine = RidgeCV(alphas=alphas2, store_cv_values=True)
ridge_fine.fit(X_train_scaled, y_train_prod['unit_target'])
print(f"Best alpha from second search: {ridge_fine.alpha_}")